In [ ]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image
import torch
from ultralytics import YOLO

## 1. Project paths

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

def find_project_root(start=None):
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not find project root.")

PROJECT_ROOT = find_project_root()
REPORT_DIR = PROJECT_ROOT / "reports" / "damage_error_analysis"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

## 2. Locating CarDD and the checkpoint

In [ ]:
DATASET_CANDIDATES = [
    PROJECT_ROOT / "data" / "raw" / "cardd_kaggle" / "extracted",
    PROJECT_ROOT / "data" / "external" / "cardd_kaggle" / "extracted",
    Path(
        r"G:\.shortcut-targets-by-id\1OOvt005TduvEA8h6_snfuyQtbQAjo-3J"
        r"\capstone-vehicle-metadata\data\cardd_kaggle\extracted"
    ),
]

DATASET_ROOT = next(
    (
        p for p in DATASET_CANDIDATES
        if (p / "val" / "images").exists()
        and (p / "val" / "labels").exists()
    ),
    None,
)

CHECKPOINT = (
    PROJECT_ROOT
    / "models"
    / "damage_detection_benchmark"
    / "yolo11s_cardd_640px"
    / "weights"
    / "best.pt"
)

print("DATASET_ROOT:", DATASET_ROOT)
print("CHECKPOINT:", CHECKPOINT)

if DATASET_ROOT is None:
    raise FileNotFoundError("Could not locate CarDD.")
if not CHECKPOINT.exists():
    raise FileNotFoundError("Could not locate the YOLO11s best.pt checkpoint.")

## 3. Settings

In [ ]:
CLASS_NAMES = [
    "dent",
    "scratch",
    "crack",
    "glass shatter",
    "lamp broken",
    "tire flat",
]

FOCUS_CLASSES = {"dent", "scratch", "crack"}

CONF_THRESHOLD = 0.25
IOU_MATCH_THRESHOLD = 0.50
IMGSZ = 640
DEVICE = 0 if torch.cuda.is_available() else "cpu"

model = YOLO(str(CHECKPOINT))
print(model.names)

## 4. Ground-truth loader

In [ ]:
def yolo_xywh_to_xyxy(xc, yc, w, h, image_width, image_height):
    xc *= image_width
    yc *= image_height
    w *= image_width
    h *= image_height
    return [
        xc - w / 2,
        yc - h / 2,
        xc + w / 2,
        yc + h / 2,
    ]

def load_ground_truth(image_path):
    image_path = Path(image_path)
    label_path = DATASET_ROOT / "val" / "labels" / f"{image_path.stem}.txt"

    with Image.open(image_path) as img:
        image_width, image_height = img.size

    rows = []

    if not label_path.exists():
        return pd.DataFrame(
            columns=["class_id", "class_name", "x1", "y1", "x2", "y2"]
        )

    for line in label_path.read_text().splitlines():
        parts = line.strip().split()
        if len(parts) < 5:
            continue

        class_id = int(parts[0])
        xc, yc, w, h = map(float, parts[1:5])
        x1, y1, x2, y2 = yolo_xywh_to_xyxy(
            xc, yc, w, h, image_width, image_height
        )

        rows.append({
            "class_id": class_id,
            "class_name": CLASS_NAMES[class_id],
            "x1": x1,
            "y1": y1,
            "x2": x2,
            "y2": y2,
        })

    return pd.DataFrame(rows)

## 5. IoU helper

In [ ]:
def box_iou(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b

    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)

    iw = max(0.0, ix2 - ix1)
    ih = max(0.0, iy2 - iy1)
    intersection = iw * ih

    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)

    union = area_a + area_b - intersection
    return 0.0 if union <= 0 else intersection / union

## 6. Predicting on validation images

In [ ]:
val_image_dir = DATASET_ROOT / "val" / "images"

image_extensions = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp",
}

val_images = sorted(
    p
    for p in val_image_dir.iterdir()
    if p.suffix.lower() in image_extensions
)

print("Validation images:", len(val_images))

if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 7. Matching predictions to ground truth

In [ ]:
analysis_rows = []
extra_rows = []

for i, image_path in enumerate(val_images):

    result = model.predict(
        source=str(image_path),
        imgsz=IMGSZ,
        conf=CONF_THRESHOLD,
        device=DEVICE,
        verbose=False,
    )[0]

    if (i + 1) % 50 == 0:
        print(f"Processed {i + 1}/{len(val_images)} images")
    gt_df = load_ground_truth(image_path)

    pred_rows = []
    if result.boxes is not None and len(result.boxes) > 0:
        for idx, (box, cls_id, conf) in enumerate(
            zip(
                result.boxes.xyxy.cpu().numpy(),
                result.boxes.cls.cpu().numpy().astype(int),
                result.boxes.conf.cpu().numpy(),
            )
        ):
            pred_rows.append({
                "pred_idx": idx,
                "class_name": model.names[int(cls_id)],
                "confidence": float(conf),
                "x1": float(box[0]),
                "y1": float(box[1]),
                "x2": float(box[2]),
                "y2": float(box[3]),
            })

    pred_df = pd.DataFrame(pred_rows)
    used = set()

    for gt_idx, gt in gt_df.iterrows():
        gt_box = [gt.x1, gt.y1, gt.x2, gt.y2]

        best_iou = 0.0
        best_pred_idx = None
        best_pred = None

        for pred_idx, pred in pred_df.iterrows():
            if pred_idx in used:
                continue

            pred_box = [pred.x1, pred.y1, pred.x2, pred.y2]
            iou = box_iou(gt_box, pred_box)

            if iou > best_iou:
                best_iou = iou
                best_pred_idx = pred_idx
                best_pred = pred

        if best_pred is None or best_iou < IOU_MATCH_THRESHOLD:
            status = "missed"
            pred_class = None
            pred_conf = np.nan
        else:
            used.add(best_pred_idx)
            pred_class = best_pred.class_name
            pred_conf = best_pred.confidence
            status = "correct" if pred_class == gt.class_name else "wrong_class"

        analysis_rows.append({
            "image_name": image_path.name,
            "image_path": str(image_path),
            "gt_index": int(gt_idx),
            "gt_class": gt.class_name,
            "pred_class": pred_class,
            "pred_confidence": pred_conf,
            "best_iou": best_iou,
            "status": status,
        })

    for pred_idx, pred in pred_df.iterrows():
        if pred_idx not in used:
            extra_rows.append({
                "image_name": image_path.name,
                "image_path": str(image_path),
                "pred_class": pred.class_name,
                "pred_confidence": pred.confidence,
                "status": "extra_prediction",
            })

    del result

    if torch.cuda.is_available() and (i + 1) % 50 == 0:
        torch.cuda.empty_cache()

analysis_df = pd.DataFrame(analysis_rows)
extra_predictions_df = pd.DataFrame(extra_rows)

display(analysis_df.head())

## 8. Error summary by class

In [ ]:
summary = (
    analysis_df.groupby(["gt_class", "status"])
    .size()
    .unstack(fill_value=0)
)

for col in ["correct", "missed", "wrong_class"]:
    if col not in summary.columns:
        summary[col] = 0

summary["total"] = summary[["correct", "missed", "wrong_class"]].sum(axis=1)
summary["correct_rate"] = summary["correct"] / summary["total"]
summary["miss_rate"] = summary["missed"] / summary["total"]
summary["wrong_class_rate"] = summary["wrong_class"] / summary["total"]

display(summary.sort_values("correct_rate"))

analysis_df.to_csv(
    REPORT_DIR / "validation_gt_match_analysis.csv",
    index=False,
)
extra_predictions_df.to_csv(
    REPORT_DIR / "validation_extra_predictions.csv",
    index=False,
)
summary.to_csv(
    REPORT_DIR / "validation_error_summary_by_class.csv"
)

## 9. Crack, dent and scratch focus

In [ ]:
focus_df = analysis_df[
    analysis_df["gt_class"].isin(FOCUS_CLASSES)
].copy()

display(
    focus_df.groupby(["gt_class", "status"])
    .size()
    .unstack(fill_value=0)
)

display(
    focus_df[
        focus_df["status"] == "wrong_class"
    ]
    .groupby(["gt_class", "pred_class"])
    .size()
    .sort_values(ascending=False)
    .to_frame("count")
)

## 10. Ground truth vs prediction visualiser

In [ ]:
def draw_boxes(ax, image, boxes_df, title, prediction=False):
    ax.imshow(image)
    ax.set_title(title)
    ax.axis("off")

    for _, row in boxes_df.iterrows():
        x1, y1, x2, y2 = row["x1"], row["y1"], row["x2"], row["y2"]

        ax.add_patch(
            Rectangle(
                (x1, y1),
                x2 - x1,
                y2 - y1,
                fill=False,
                linewidth=2,
            )
        )

        label = row["class_name"]

        if prediction:
            label += f" {row['confidence']:.2f}"

        ax.text(
            x1,
            max(0, y1 - 3),
            label,
            fontsize=9,
            bbox={"facecolor": "white", "alpha": 0.7},
        )

def show_gt_vs_prediction(image_path):
    image_path = Path(image_path)
    image = Image.open(image_path).convert("RGB")

    gt_df = load_ground_truth(image_path)

    result = model.predict(
        source=str(image_path),
        imgsz=IMGSZ,
        conf=CONF_THRESHOLD,
        device=DEVICE,
        verbose=False,
    )[0]

    pred_rows = []

    if result.boxes is not None:
        for box, cls_id, conf in zip(
            result.boxes.xyxy.cpu().numpy(),
            result.boxes.cls.cpu().numpy().astype(int),
            result.boxes.conf.cpu().numpy(),
        ):
            pred_rows.append({
                "class_name": model.names[int(cls_id)],
                "confidence": float(conf),
                "x1": float(box[0]),
                "y1": float(box[1]),
                "x2": float(box[2]),
                "y2": float(box[3]),
            })

    pred_df = pd.DataFrame(pred_rows)

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    draw_boxes(axes[0], image, gt_df, "Ground truth")
    draw_boxes(axes[1], image, pred_df, "YOLO11s prediction", prediction=True)

    fig.suptitle(image_path.name)
    plt.tight_layout()
    plt.show()

## 11. Inspecting difficult examples

In [ ]:
for focus_class in ["crack", "dent", "scratch"]:
    difficult = (
        focus_df[
            (focus_df["gt_class"] == focus_class)
            & (focus_df["status"] != "correct")
        ]
        .drop_duplicates("image_path")
    )

    print("\n" + "=" * 80)
    print(f"{focus_class.upper()} difficult examples: {len(difficult)}")
    print("=" * 80)

    if len(difficult) == 0:
        continue

    examples = difficult.sample(
        n=min(4, len(difficult)),
        random_state=SEED,
    )

    for image_path in examples["image_path"]:
        show_gt_vs_prediction(image_path)